In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.window import Window

##Load all required tables

In [0]:
df_silver_orders = spark.table(SILVER_ORDERS)
df_dim_city = spark.table(GOLD_CITY)
df_dim_payment_method = spark.table(GOLD_PAYMENT_METHOD)
df_dim_device_type = spark.table(GOLD_DEVICE_TYPE)

display(df_silver_orders)

##Build fact_weather

In [0]:
df_fact_orders = (df_silver_orders
   # Join dim_city on city
   .join(
       df_dim_city.select("city_id", "city"),
       on="city",
       how="left"
   )
   # Join dim_payment_method on payment_method
   .join(
       df_dim_payment_method.select("payment_method_id", "payment_method"),
       on="payment_method",
       how="left"
   )
   # Join dim_device_type on device_type
   .join(
       df_dim_device_type.select("device_type_id", "device_type"),
       on="device_type",
       how="left"
   )
   # Compute total amount
   .withColumn("total_amount",
       F.round(F.col("quantity") * F.col("unit_price_sar"), 2)
   )
   # Ingestion metadata
   .withColumn("_gold_ingested_at",
       F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
   )
   .select(
       "order_id",
       "customer_id",
       "product_id",
       "city_id",
       "payment_method_id",
       "device_type_id",
       "order_date",
       "quantity",
       "unit_price_sar",
       "total_amount",
       "order_status",
       "_gold_ingested_at"
   )
)
print(f"fact_orders rows: {df_fact_orders.count()}")
display(df_fact_orders)

##Validate join quality

In [0]:
print(f"Null customer_id      : {df_fact_orders.filter(F.col('customer_id').isNull()).count()}")
print(f"Null city_id          : {df_fact_orders.filter(F.col('city_id').isNull()).count()}")
print(f"Null payment_method_id: {df_fact_orders.filter(F.col('payment_method_id').isNull()).count()}")
print(f"Null device_type_id   : {df_fact_orders.filter(F.col('device_type_id').isNull()).count()}")

##Write to gold

In [0]:
(df_fact_orders
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(GOLD_ORDERS)
)
print(f"✅ {df_fact_orders.count()} rows written to {GOLD_ORDERS}")

##Validate

In [0]:
df_check = spark.table(GOLD_ORDERS)
print(f"Total rows          : {df_check.count()}")
print(f"Distinct orders     : {df_check.select('order_id').distinct().count()}")
print(f"Total revenue SAR   : {df_check.agg(F.round(F.sum('total_amount'), 2)).collect()[0][0]}")
print(f"Avg order value SAR : {df_check.agg(F.round(F.avg('total_amount'), 2)).collect()[0][0]}")
print(f"Order statuses      : {[r[0] for r in df_check.select('order_status').distinct().collect()]}")
df_check.show(5, truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.fact_orders;